# Week 1 - Image Formation as Arrays

This notebook accompanies the first MATH 435 slide deck.

## Goal

By the end, you should be able to:

- read an image as a numerical array;
- inspect shape, intensity range, and individual pixels;
- flatten an image into a vector and reshape it back;
- model simple missing-pixel observations with a mask;
- explain why masking recorded pixels is not the same as undoing scene occlusion.

## Setup

The notebook uses NumPy, scikit-image, and Plotly.

If you run this outside Google Colab and a package is missing, install the course requirements from the repository root:

```bash
python3 -m pip install -r requirements.txt
```

In [ ]:
import numpy as np
from skimage import data
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def show_gray(image, title="", zmin=None, zmax=None, height=450, text_auto=False):
    fig = px.imshow(
        image,
        color_continuous_scale="gray",
        origin="upper",
        aspect="equal",
        zmin=zmin,
        zmax=zmax,
        text_auto=text_auto,
        binary_string=False,
    )
    fig.update_layout(
        title=title,
        height=height,
        margin=dict(l=20, r=20, t=55, b=20),
    )
    fig.update_coloraxes(colorbar_title="intensity")
    fig.show()


def show_image_grid(images, titles, zmin=0, zmax=1, height=430):
    fig = make_subplots(rows=1, cols=len(images), subplot_titles=titles)
    for index, image in enumerate(images, start=1):
        fig.add_trace(
            go.Heatmap(
                z=image,
                colorscale="Gray",
                zmin=zmin,
                zmax=zmax,
                showscale=index == len(images),
            ),
            row=1,
            col=index,
        )
        fig.update_xaxes(showticklabels=False, row=1, col=index)
        fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=index)
    fig.update_layout(height=height, margin=dict(l=20, r=20, t=60, b=20))
    fig.show()

## Step 1 - A Tiny Image Array

A grayscale image can be represented by a matrix of intensity values.

In [ ]:
small_image = np.arange(20).reshape(4, 5)

print("shape:", small_image.shape)
print(small_image)

show_gray(
    small_image,
    title="A 4 by 5 image array",
    zmin=small_image.min(),
    zmax=small_image.max(),
    height=350,
    text_auto=True,
)

### Exercise 1

Change the number of rows and columns below. Before running the cell, predict:

- the shape of the array;
- the first row;
- the last value.

In [ ]:
# TODO: change these values.
rows = 6
cols = 6

exercise_image = np.arange(rows * cols).reshape(rows, cols)

print("shape:", exercise_image.shape)
print("first row:", exercise_image[0])
print("last value:", exercise_image[-1, -1])
show_gray(exercise_image, title="Exercise image", height=380, text_auto=True)

## Step 2 - Vectorizing an Image

Many imaging models write the unknown image as a vector `x`.
Vectorization does not change the information; it changes the shape.

In [ ]:
vector = small_image.reshape(-1)
restored = vector.reshape(small_image.shape)

print("vector shape:", vector.shape)
print(vector)
print("restored equals original:", np.array_equal(restored, small_image))

### Exercise 2

For row-major vectorization, the pixel at row `r` and column `c` goes to index

```text
r * number_of_columns + c
```

Check this formula below.

In [ ]:
# TODO: change row and col, then compare the matrix value and vector value.
row = 2
col = 3

flat_index = row * small_image.shape[1] + col

print("matrix value:", small_image[row, col])
print("flat index:", flat_index)
print("vector value:", vector[flat_index])

## Step 3 - A Real Image

We now load a real grayscale image from `skimage.data`.
Intensities are scaled to the interval `[0, 1]`.

In [ ]:
image = data.camera().astype(float) / 255.0

print("shape:", image.shape)
print("dtype:", image.dtype)
print("min:", image.min())
print("max:", image.max())

show_gray(image, title="Real grayscale image as an array", zmin=0, zmax=1, height=520)

## Step 4 - Looking at Pixels

A small crop lets us inspect individual intensity values.

In [ ]:
# TODO: change r0, c0, and size to inspect another part of the image.
r0 = 145
c0 = 150
size = 10

crop = image[r0 : r0 + size, c0 : c0 + size]

print("crop shape:", crop.shape)
show_gray(
    np.round(crop, 2),
    title="Zoomed crop with intensity values",
    zmin=0,
    zmax=1,
    height=460,
    text_auto=".2f",
)

## Step 5 - A Simple Observation Mask

A mask can model which recorded pixels are available. Here we remove a rectangular region from the recorded image.

In [ ]:
mask = np.ones_like(image, dtype=bool)
mask[120:310, 160:320] = False

masked_observation = image.copy()
masked_observation[~mask] = 0.0

show_image_grid(
    [image, masked_observation],
    ["original image", "masked observation"],
    zmin=0,
    zmax=1,
    height=500,
)

flat_image = image.reshape(-1)
flat_mask = mask.reshape(-1)
observations = flat_image[flat_mask]

print("unknown image length:", flat_image.size)
print("observed pixel count:", observations.size)
print("observed fraction:", round(float(flat_mask.mean()), 3))

### Exercise 3

Change the mask rule below. Try observing every second, fourth, or tenth pixel.

What changes in the observation vector?

In [ ]:
# TODO: change observed_every.
observed_every = 4

toy_vector = small_image.reshape(-1)
toy_mask = np.arange(toy_vector.size) % observed_every != 0
toy_observations = toy_vector[toy_mask]

print("toy vector:", toy_vector)
print("mask:", toy_mask.astype(int))
print("observations:", toy_observations)
print("observed fraction:", round(float(toy_mask.mean()), 3))

## Checkpoint

Answer in your own words:

1. What is the difference between an image array and an image vector?
2. What information does `image.shape` give?
3. What does a binary mask select?
4. Why is a missing recorded pixel not the same as seeing behind an occluding object?